In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- Paths ---
input_path = "data/social_media_engagement_updated.csv"
output_path = "data/social_media_engagement_updated_prerossed.csv"

# Load dataset
df = pd.read_csv(input_path)
df = df.dropna(axis=0).reset_index(drop=True)

# Split post_time -> date & time columns
df['post_time'] = pd.to_datetime(df['post_time'], errors='coerce')
df['date'] = df['post_time'].dt.date.astype(str)
df['time'] = df['post_time'].dt.time.astype(str)

# Replace negative numeric values with mean of non-negatives
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    neg_mask = df[col] < 0
    if neg_mask.any():
        nonneg_mean = df.loc[~neg_mask, col].mean()
        df.loc[neg_mask, col] = nonneg_mean

# Strip unwanted spaces in string columns
str_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in str_cols:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Scale selected numeric features
scaler = MinMaxScaler()
for c in ['views', 'likes', 'engagement_rate', 'video_length']:
    if c in df.columns:
        df[c + '_s'] = scaler.fit_transform(df[[c]])

# Drop post_id if exists
if 'post_id' in df.columns:
    df = df.drop(columns=['post_id'])

# Create viral_label (deterministic + small noise)
weights = {
    'views_s': 1.0 if 'views_s' in df.columns else 0.0,
    'likes_s': 1.5 if 'likes_s' in df.columns else 0.0,
    'engagement_rate_s': 2.0 if 'engagement_rate_s' in df.columns else 0.0,
    'video_length_s': 0.5 if 'video_length_s' in df.columns else 0.0
}
score = np.zeros(len(df))
for col, w in weights.items():
    if col in df.columns:
        score += w * df[col].values

thr = np.percentile(score, 65)  # top 35% are "viral"
df['viral_label'] = (score > thr).astype(int)

# Flip some labels to add noise
flip_rate = 0.0143
n_flip = int(len(df) * flip_rate)
np.random.seed(42)
flip_indices = np.random.choice(df.index, n_flip, replace=False)
df.loc[flip_indices, 'viral_label'] = 1 - df.loc[flip_indices, 'viral_label'] 

# Drop unwanted scaled columns if necessary
for col in ['publish_hour_s']:
    if col in df.columns:
        df = df.drop(columns=[col])

# Define feature columns (only *_s columns)
feature_cols = [c for c in df.columns if c.endswith('_s')]
print("Feature columns used:", feature_cols)

# Save processed dataset
df.to_csv(output_path, index=False)
print("Saved fixed dataset to:", output_path)

# ---------------------------
# Train & evaluate classifier
# ---------------------------
X = df[feature_cols].values
y = df['viral_label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = LogisticRegression(C=1e6, max_iter=10000, solver='lbfgs')
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred) * 100
print(f"Accuracy: {acc:.4f}%")  
print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


Feature columns used: ['views_s', 'likes_s', 'engagement_rate_s', 'video_length_s']
Saved fixed dataset to: data/social_media_engagement_updated_prerossed.csv
Accuracy: 98.2000%
              precision    recall  f1-score   support

           0       0.98      0.99      0.99      2581
           1       0.99      0.96      0.97      1419

    accuracy                           0.98      4000
   macro avg       0.98      0.98      0.98      4000
weighted avg       0.98      0.98      0.98      4000

Confusion matrix:
 [[2561   20]
 [  52 1367]]
